In [1]:
from pathlib import Path
from concurrent .futures import ThreadPoolExecutor
import os

import numpy as np
import pandas as pd
import geopandas as gpd

import matplotlib .pyplot as plt

import mlx .core as mx

from tqdm import tqdm


In [2]:
hourly_hrrr_data_base_directory =Path ('/Volumes/Hot Cache/temporal_variables_weather_partial/')

weather_db_directory =Path ('/Users/aaronspaulding/data/weather_db_hurricane_regions/')


In [3]:
relevant_storms =gpd .read_feather ('data_cache/relevant_storm_tracks.feather')
relevant_storms =relevant_storms [::-1 ].reset_index (drop =True )


In [5]:
h3_cells =gpd .read_feather (weather_db_directory /'h3_cells_cached.feather')


In [6]:
max_workers =min (32 ,max (4 ,(os .cpu_count ()or 8 )*2 ))

h3_index =h3_cells ['index'].to_numpy ()

def _output_paths (file_name :str ):
    stem =file_name .removesuffix ('.feather')
    return (
    weather_db_directory /'accumulated_precipitation'/f'{stem}.npy',
    weather_db_directory /'gust_speed'/f'{stem}.npy',
    weather_db_directory /'wind_speed'/f'{stem}.npy',
    )

def _outputs_exist (file_name :str )->bool :
    out_accum ,out_gust ,out_wind =_output_paths (file_name )
    return out_accum .exists ()and out_gust .exists ()and out_wind .exists ()

def _process_hour_file (file_name :str )->str :
    out_accum ,out_gust ,out_wind =_output_paths (file_name )

    if out_accum .exists ()and out_gust .exists ()and out_wind .exists ():
        return 'skipped'

    hrrr_file_path =hourly_hrrr_data_base_directory /file_name

    try :
        hrrr_data =pd .read_feather (hrrr_file_path )
    except FileNotFoundError :
        return 'missing_hrrr'

    hrrr_data =hrrr_data .set_index ('index')

    try :
        ordered =hrrr_data .loc [h3_index ,['wind_speed','gust_speed','accumulated_precipitation']]
    except KeyError :
        return 'missing_cells'

    wind_speed_data =mx .array (ordered ['wind_speed'].to_numpy (),dtype =mx .float16 )
    gust_speed_data =mx .array (ordered ['gust_speed'].to_numpy (),dtype =mx .float16 )
    accumulated_precipitation_data =mx .array (ordered ['accumulated_precipitation'].to_numpy (),dtype =mx .float16 )

    mx .save (out_wind ,wind_speed_data )
    mx .save (out_gust ,gust_speed_data )
    mx .save (out_accum ,accumulated_precipitation_data )

    return 'processed'

for j ,storm_row in tqdm (relevant_storms .iterrows (),total =relevant_storms .shape [0 ]):
    storm_id =storm_row ['SID']
    storm_name =storm_row ['NAME']
    storm_start =storm_row ['datetime_min'].floor ('h')
    storm_end =storm_row ['datetime_max'].ceil ('h')+pd .to_timedelta (1 ,'d')

    dates =pd .date_range (storm_start ,storm_end ,freq ='h')
    dates =pd .DataFrame (dates ,columns =['datetime'])
    dates ['file_name']=dates ['datetime'].dt .strftime ('%Y_%m_%dT%H_%M_%S')+'.feather'
    dates ['file_exists']=dates ['file_name'].apply (lambda file_name :(hourly_hrrr_data_base_directory /file_name ).exists ())
    dates =dates .sort_values ('datetime').reset_index (drop =True )

    number_of_missing_hrrr_files =np .sum (dates ['file_exists']==False )
    relevant_storms .loc [j ,'number_of_missing_files']=number_of_missing_hrrr_files

    if number_of_missing_hrrr_files >0 :
        continue

    dates ['processed_exists']=dates ['file_name'].apply (_outputs_exist )
    pending_file_names =dates .loc [~dates ['processed_exists'],'file_name'].drop_duplicates ().tolist ()

    if not pending_file_names :
        relevant_storms .loc [j ,'weather_files_processed']=0
        relevant_storms .loc [j ,'weather_files_skipped']=int (dates .shape [0 ])
        continue

    with ThreadPoolExecutor (max_workers =max_workers )as pool :
        results =list (
        tqdm (
        pool .map (_process_hour_file ,pending_file_names ),
        total =len (pending_file_names ),
        leave =False ,
        desc =f'{storm_name} {storm_id}',
        )
        )

    processed_count =sum (r =='processed'for r in results )
    skipped_count =sum (r =='skipped'for r in results )
    missing_hrrr_count =sum (r =='missing_hrrr'for r in results )
    missing_cells_count =sum (r =='missing_cells'for r in results )

    relevant_storms .loc [j ,'weather_files_processed']=int (processed_count )
    relevant_storms .loc [j ,'weather_files_skipped']=int (skipped_count )
    relevant_storms .loc [j ,'weather_files_missing_hrrr']=int (missing_hrrr_count )
    relevant_storms .loc [j ,'weather_files_missing_cells']=int (missing_cells_count )


100%|██████████| 81/81 [40:17<00:00, 29.85s/it]                         
